# XGBoost - final model for SAFEX white maize

Gradient-boosted trees on lagged prices and lagged external drivers (**ZAR/USD, rainfall, fuel price**).
The original soybean project finished with XGBoost because it explains itself: the feature-importance
ranking shows *which* drivers matter. Whether it is also the most accurate model for white maize is for
the comparison table in the README to decide.

Design choices (all easy to change in the settings cell):

* **No look-ahead.** Every feature is lagged by at least `HORIZON` business days, so the model only uses
  information that would have been available when the forecast was made.
* **Predict the price change, not the price level** (`PREDICT = 'diff'`). Trees cannot predict values outside
  the range they were trained on, which is a problem when maize prices trend to new highs. The predicted
  change is added to the last known price, and all scores are reported in R/ton.
* **Time-aware tuning.** `TimeSeriesSplit` replaces shuffled K-fold, which leaks the future into the past.
* **Chronological 80 / 20 split**, identical to the other notebooks.

## Setup

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn import metrics
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit

warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
# ---- Data settings ---------------------------------------------------------
PRICES_FILE = 'safex_prices.csv'   # Date, Close (R/ton)  [+ optional Open, High, Low, Volume]
EXOG_FILE   = 'sa_factors.csv'     # Date, ZAR_USD, Rainfall_mm, Fuel_Price
TARGET_COL  = 'Close'              # SAFEX white maize price column
EXOG_COLS   = ['ZAR_USD', 'Rainfall_mm', 'Fuel_Price']   # external drivers
TRAIN_FRAC  = 0.8                  # chronological split: first 80% train, last 20% test

# ---- Model settings --------------------------------------------------------
HORIZON     = 1                    # forecast horizon in business days (1 = next day, 5 = one week ahead)
PREDICT     = 'diff'               # 'diff' = predict the change vs the last known price; 'level' = the price itself
FEATURE_KW  = dict(lag_prices=10, lag_exog=5)   # how many lagged days of price changes / of each driver
N_ITER      = 100                  # random-search iterations (lower it for a quick test run)

## Define necessary functions

### Data preparation

Loads both files, merges them with an inner join and puts them on a business-day calendar. Lower-frequency
drivers (e.g. monthly fuel prices) are forward-filled to daily **before** the join so that the inner join
does not discard the daily price rows.

In [ ]:
def prepare_data(prices_file=PRICES_FILE, exog_file=EXOG_FILE):
    prices = pd.read_csv(prices_file, parse_dates=['Date'], index_col='Date').sort_index()
    exog_data = pd.read_csv(exog_file, parse_dates=['Date'], index_col='Date').sort_index()
    prices = prices[~prices.index.duplicated(keep='last')]
    exog_data = exog_data[~exog_data.index.duplicated(keep='last')]

    missing = [c for c in [TARGET_COL] if c not in prices.columns] + \
              [c for c in EXOG_COLS if c not in exog_data.columns]
    if missing:
        raise KeyError(f'Column(s) not found in the CSV files: {missing}')

    exog_data = exog_data.resample('D').last().ffill()   # monthly / weekly drivers -> daily, last value carried forward
    data = prices.join(exog_data, how='inner')           # align the dates
    data = data.resample('B').ffill()                    # business days, forward-fill gaps
    return data.dropna(subset=[TARGET_COL] + EXOG_COLS)

### Feature engineering

Every feature for the price on day *t* is built from information up to day *t - HORIZON*:

* the last known price, lagged **price changes**, distance from the 5- / 20-day moving average and recent volatility
* lagged values of every external driver
* the daily high-low range, if your price file has `High` and `Low`
* calendar features (month, week of year, weekday) to capture the harvest cycle

In [ ]:
def build_features(data, horizon=HORIZON, lag_prices=10, lag_exog=5, ma_windows=(5, 20)):
    """Return X (features), y (target) and base (last known price when the forecast is made)."""
    close = data[TARGET_COL]
    d1 = close.diff()
    feats = {}

    # --- price memory (all shifted by >= horizon) -----------------------------
    feats[f'Close (t-{horizon})'] = close.shift(horizon)
    for k in range(horizon, horizon + lag_prices):
        feats[f'dClose (t-{k})'] = d1.shift(k)
    for w in ma_windows:
        feats[f'Close_minus_MA{w}'] = close.shift(horizon) - close.shift(horizon).rolling(w).mean()
        feats[f'Volatility{w}'] = d1.shift(horizon).rolling(w).std()

    # --- daily trading range (optional, mirrors the old 'max-min' feature) -----
    if {'High', 'Low'}.issubset(data.columns):
        day_range = data['High'] - data['Low']
        for k in range(horizon, horizon + lag_exog):
            feats[f'max-min (t-{k})'] = day_range.shift(k)

    # --- external drivers -----------------------------------------------------
    for col in EXOG_COLS:
        for k in range(horizon, horizon + lag_exog):
            feats[f'{col} (t-{k})'] = data[col].shift(k)

    # --- calendar (known in advance, so no lag needed) ---------------------------
    feats['Month'] = pd.Series(data.index.month, index=data.index)
    feats['WeekOfYear'] = pd.Series(data.index.isocalendar().week.astype(int).to_numpy(), index=data.index)
    feats['DayOfWeek'] = pd.Series(data.index.dayofweek, index=data.index)

    X = pd.DataFrame(feats, index=data.index)
    base = close.shift(horizon)                               # last price known at forecast time
    y = (close - base) if PREDICT == 'diff' else close
    return X, y, base

### Chronological split, tuning and evaluation

The split is by **date** (the same first test date as the other notebooks), so different feature settings are
always scored on exactly the same test days.

In [ ]:
def chrono_split(X, y, base, split_date):
    """Drop rows with missing features, then split at `split_date` (first test day)."""
    ok = X.notna().all(axis=1) & y.notna()
    X, y, base = X[ok], y[ok], base[ok]
    is_train = X.index < split_date
    return X[is_train], X[~is_train], y[is_train], y[~is_train], base[is_train], base[~is_train]


PARAM_DIST = {
    'learning_rate': [0.1, 0.05, 0.01],
    'n_estimators': list(range(50, 500, 25)),
    'max_depth': list(range(2, 11)),
    'min_child_weight': list(range(1, 6)),
    'gamma': [i / 10.0 for i in range(0, 5)],
    'subsample': [i / 10.0 for i in range(6, 10)],
    'colsample_bytree': [i / 10.0 for i in range(6, 10)],
    'reg_alpha': [1e-5, 1e-2, 0.1, 1, 100],
}


def tune_xgb(X_train, y_train, n_iter=N_ITER, n_splits=3, seed=45):
    """Randomised hyper-parameter search with time-ordered cross-validation (no shuffling)."""
    gbm = xgb.XGBRegressor(objective='reg:squarederror', n_jobs=1, random_state=123)
    search = RandomizedSearchCV(estimator=gbm, param_distributions=PARAM_DIST, n_iter=n_iter,
                                scoring='neg_mean_absolute_error',
                                cv=TimeSeriesSplit(n_splits=n_splits),
                                verbose=1, n_jobs=-1, random_state=seed)
    search.fit(X_train, y_train)
    return search

In [ ]:
def evaluate(search, splits, price):
    """Score a fitted search object. Everything is converted back to R/ton first."""
    X_tr, X_te, y_tr, y_te, base_tr, base_te = splits
    best = search.best_estimator_

    def to_price(pred, base):
        return pred + base.to_numpy() if PREDICT == 'diff' else pred

    pred_tr = pd.Series(to_price(best.predict(X_tr), base_tr), index=X_tr.index)
    pred_te = pd.Series(to_price(best.predict(X_te), base_te), index=X_te.index)
    act_tr, act_te = price.loc[X_tr.index], price.loc[X_te.index]

    def metrics_for(actual, pred):
        err = actual - pred
        return {'R2': metrics.r2_score(actual, pred),
                'RMSE': float(np.sqrt(np.mean(err ** 2))),
                'MAE': float(np.mean(np.abs(err))),
                'MAPE %': float(np.mean(np.abs(err / actual)) * 100)}

    return {'train': metrics_for(act_tr, pred_tr),
            'test': metrics_for(act_te, pred_te),
            'naive_test_MAE': float(np.mean(np.abs(act_te - base_te))),   # "price stays as last known"
            'pred_test': pred_te, 'actual_test': act_te, 'base_test': base_te}


def feature_importance(search, columns):
    """Gain-based importance for every feature (0 for features the model never used)."""
    booster = search.best_estimator_.get_booster()
    score = pd.Series(booster.get_score(importance_type='gain'), dtype=float)
    return score.reindex(columns).fillna(0.0).sort_values(ascending=False)

# Execution

In [ ]:
data = prepare_data()
price = data[TARGET_COL]
split_date = data.index[int(len(data) * TRAIN_FRAC)]        # first day of the test block

print(f'{len(data)} business days   {data.index.min().date()} -> {data.index.max().date()}')
print('test block starts', split_date.date())
data.tail()

In [ ]:
X, y, base = build_features(data, horizon=HORIZON, **FEATURE_KW)
splits = chrono_split(X, y, base, split_date)
X_train, X_test, y_train, y_test, base_train, base_test = splits

print(f'{X.shape[1]} features | train rows: {len(X_train)} | test rows: {len(X_test)}')
print(list(X.columns))
X.dropna().head()

### Tune and fit

Heads-up: with the default `N_ITER = 100` and 3 time-ordered folds this runs 300 model fits.

In [ ]:
search = tune_xgb(X_train, y_train)
print('best params:\n', search.best_params_)
print('Cross-validated MAE on the training block (R/ton):', round(-search.best_score_, 2))

In [ ]:
res = evaluate(search, splits, price)

print(pd.DataFrame({'train': res['train'], 'test': res['test']}).round(3))
print()
print(f"MAE of the naive 'price stays as last known' benchmark on the same test days: "
      f"R {res['naive_test_MAE']:,.2f} per ton")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(price.loc[:split_date], label='training')
plt.plot(res['actual_test'], label='actual')
plt.plot(res['pred_test'], label=f'XGBoost ({HORIZON}-day ahead)')
plt.title('XGBoost: predictions vs actuals on the test block')
plt.ylabel('R / ton')
plt.legend(loc='upper left', fontsize=8)
plt.show()

## Which drivers matter?

Gain-based feature importance: the average improvement in accuracy brought by splits on that feature.
Look at the lagged `ZAR_USD`, `Rainfall_mm` and `Fuel_Price` entries to see how much the external drivers
contribute relative to the price's own history and the calendar.

In [ ]:
feat_imp = feature_importance(search, X_train.columns)

feat_imp.head(20).iloc[::-1].plot(kind='barh', figsize=(9, 8), title='Top 20 feature importances (gain)')
plt.xlabel('Importance (gain)')
plt.tight_layout()
plt.show()

# Importance summed per driver (all lags of one variable combined)
group_names = EXOG_COLS + ['Close', 'dClose', 'Volatility', 'max-min', 'Month', 'WeekOfYear', 'DayOfWeek']
def group_of(name):
    for g in sorted(group_names, key=len, reverse=True):
        if name.startswith(g):
            return g
    return 'Other'

feat_imp.groupby(feat_imp.index.map(group_of)).sum().sort_values(ascending=False).round(3)

In [ ]:
feat_imp.to_csv('feature_importances_white_maize.csv')
search.best_estimator_.save_model('xgb_white_maize.json')

## Forecast the next price

The last row of the data becomes the "known" day, and the features for the target date (`HORIZON` business
days later) are built from it. Nothing from the future is needed because every feature is lagged by at least
`HORIZON` days.

In [ ]:
def forecast_next(search, data, columns, horizon=HORIZON, **feature_kw):
    """Forecast the price `horizon` business days after the last row of `data`."""
    future_idx = pd.bdate_range(data.index[-1] + pd.offsets.BDay(1), periods=horizon)
    data_ext = data.reindex(data.index.append(future_idx))            # empty rows for the target dates
    X_ext, _, base_ext = build_features(data_ext, horizon=horizon, **feature_kw)

    x_next = X_ext.iloc[[-1]][list(columns)]
    change_or_level = float(search.best_estimator_.predict(x_next)[0])
    price_next = base_ext.iloc[-1] + change_or_level if PREDICT == 'diff' else change_or_level
    return future_idx[-1], price_next


next_date, next_price = forecast_next(search, data, X_train.columns, horizon=HORIZON, **FEATURE_KW)
print(f'Forecast SAFEX white maize for {next_date.date()}: R {next_price:,.2f} per ton')

## Lag sensitivity (optional)

How many days of price and driver history should the model see? Each combination is tuned with a smaller
random search. Configurations are ranked by the **cross-validated** error on the training block - picking
the winner by test error would be overfitting to the test block - and the test numbers are shown only for
information.

In [ ]:
N_ITER_SMALL = 20
rows = []
for lag_prices in (5, 10, 20):
    for lag_exog in (1, 3, 5, 10):
        Xg, yg, bg = build_features(data, horizon=HORIZON, lag_prices=lag_prices, lag_exog=lag_exog)
        sp = chrono_split(Xg, yg, bg, split_date)
        s = tune_xgb(sp[0], sp[2], n_iter=N_ITER_SMALL)
        ev = evaluate(s, sp, price)
        rows.append({'PriceLag': lag_prices, 'ExogLag': lag_exog, 'CV MAE': -s.best_score_,
                     'Train R2': ev['train']['R2'], 'Test R2': ev['test']['R2'],
                     'Test RMSE': ev['test']['RMSE'], 'Test MAE': ev['test']['MAE']})

lag_table = pd.DataFrame(rows).sort_values('CV MAE').reset_index(drop=True)
lag_table.round(3)

## Profitability for the farmer: SELL or HOLD

Suppose a farmer has `TONNES` tonnes of white maize to sell. Each day the model forecasts the price
`H_SELL` business days ahead:

* forecast **above** today's price (plus the cost of holding) -> **HOLD** and sell `H_SELL` days later
* otherwise -> **SELL** today

Each test day is evaluated as an independent decision. The result is compared with *always selling today*
(zero by definition), *always holding* and a perfect-foresight upper bound. Storage and financing costs
differ between farms, so set `HOLD_COST_R_PER_TON` to your own figure (it is `0` here, which flatters HOLD).

In [ ]:
H_SELL = 5                      # decision horizon in business days (one trading week)
TONNES = 100                    # size of the farmer's stock
HOLD_COST_R_PER_TON = 0.0       # total cost of storing / financing one ton for H_SELL days (R)

# A dedicated model that forecasts H_SELL days ahead
X5, y5, base5 = build_features(data, horizon=H_SELL, **FEATURE_KW)
splits5 = chrono_split(X5, y5, base5, split_date)
search5 = tune_xgb(splits5[0], splits5[2], n_iter=max(N_ITER // 2, 10))
res5 = evaluate(search5, splits5, price)

# Row t = the day the maize would be sold if held; the decision was taken on day t - H_SELL
price_now = res5['base_test']              # price on the decision day
forecast_price = res5['pred_test']         # forecast of the price H_SELL days later
realised_price = res5['actual_test']       # what the price actually was H_SELL days later

hold = forecast_price > price_now + HOLD_COST_R_PER_TON
gain_if_hold = TONNES * (realised_price - price_now - HOLD_COST_R_PER_TON)    # vs selling today

decisions = pd.DataFrame({
    'price_today': price_now, 'forecast': forecast_price, 'realised': realised_price,
    'decision': np.where(hold, 'HOLD', 'SELL'),
    'gain_model': np.where(hold, gain_if_hold, 0.0),
    'gain_always_hold': gain_if_hold,
    'gain_perfect': np.maximum(gain_if_hold, 0.0),
})

direction_hit = ((forecast_price - price_now) > 0) == ((realised_price - price_now) > 0)
summary = pd.DataFrame({
    'total (R)': decisions[['gain_model', 'gain_always_hold', 'gain_perfect']].sum(),
    'average per day (R)': decisions[['gain_model', 'gain_always_hold', 'gain_perfect']].mean(),
}).round(0)
print(f'{TONNES} t, {H_SELL}-day horizon, {len(decisions)} test days; '
      f'direction correct on {direction_hit.mean():.0%} of days; HOLD on {hold.mean():.0%} of days')
summary

In [ ]:
decisions[['gain_model', 'gain_always_hold', 'gain_perfect']].cumsum().plot(figsize=(12, 5))
plt.title(f'Cumulative gain vs selling every day at the day-of price ({TONNES} t per decision)')
plt.ylabel('Rand')
plt.show()